# Setup & Imports

In [34]:
import sys
import numpy as np

# Add the src directory to the Python path
sys.path.append("src")

from QuantumCircuit import QuantumCircuit
from gates.registry import GateRegistry
from error_channels.default_noise import build_default_noise_model
from error_channels.ChannelRegistry import ChannelRegistry
from gates.registry import Gate

# Create Quantum Circuit

In [5]:
# Create a 2-qubit circuit
qc = QuantumCircuit(num_qubits=2, num_cbits=2, enable_metrics=True)

qc


QuantumCircuit(num_qubits=2, num_cbits=2, gates=0)

# Apply Gates (H, X, CNOT)

In [10]:
gate_reg = GateRegistry()
# Access built-in gates through registry
H = gate_reg.get("h")
X = gate_reg.get("x")
CNOT = gate_reg.get("cx")

# Apply gates
qc.add_gate(H, 0)         # Hadamard on qubit 0
qc.add_gate(X, 1)         # X on qubit 1
qc.add_gate(CNOT, [0, 1]) # CNOT: control=0, target=1

qc.ops


[('gate', <gates.registry.Gate at 0x177ce9b6dc0>, 0),
 ('gate', <gates.registry.Gate at 0x177ce49ddc0>, 1),
 ('gate', <gates.registry.Gate at 0x177ce9b6280>, [0, 1])]

# Measure Qubits

In [11]:
qc.measure(0, 0)  # measure qubit 0 into classical bit 0
qc.measure(1, 1)  # measure qubit 1 into classical bit 1

qc.ops


[('gate', <gates.registry.Gate at 0x177ce9b6dc0>, 0),
 ('gate', <gates.registry.Gate at 0x177ce49ddc0>, 1),
 ('gate', <gates.registry.Gate at 0x177ce9b6280>, [0, 1]),
 ('measure', 0, 0),
 ('measure', 1, 1)]

# Execute the Circuit

In [12]:
result_state = qc.execute(verbose=True)

print("Final state vector:")
print(result_state)

QUANTUM CIRCUIT EXECUTION RESULT

[CIRCUIT INFO]
  Qubits:       2
  Classical:    2
  Operations:   5 (3 gates, 2 measurements, 0 resets)

[CLASSICAL REGISTER]
  Bitstring: |10⟩
  c[0] = 1
  c[1] = 0

[MEASUREMENTS]
  q[0] → 1 (stored in c[0])
  q[1] → 0 (stored in c[1])

[FINAL STATE]
  |10⟩: +1.000000  (P = 1.000000)

[PERFORMANCE METRICS]
  Execution time: 0.017626s
  Peak memory:    115.92 MB
  Memory delta:   +0.22 MB


Final state vector:
{'success': True, 'state_vector': array([0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j]), 'probabilities': array([0., 0., 1., 0.]), 'classical_bits': {'c[0]': 1, 'c[1]': 0, 'bitstring': '10'}, 'measurements': {'q[0]': {'outcome': 1, 'stored_in': 'c[0]'}, 'q[1]': {'outcome': 0, 'stored_in': 'c[1]'}}, 'circuit_info': {'num_qubits': 2, 'num_cbits': 2, 'num_operations': 5, 'gate_count': 3, 'measurement_count': 2, 'reset_count': 0}, 'metrics': {'execution': {'total_time_seconds': 0.017626000000291242, 'start_time': 3049.8646443, 'end_time': 3049.8822703}, 'memory'

# Measurement Results

In [14]:
print("Measurement results:")
print(qc.get_cbits().get_bits())


Measurement results:
[1 0]


# Running Circuit Multiple Times

In [31]:
from QuantumCircuit import QuantumCircuit
from gates.registry import GateRegistry

# Set up a Bell-state circuit with 2 qubits and 2 classical bits
reg = GateRegistry()
qc_shots = QuantumCircuit(
    num_qubits=2,
    num_cbits=2,
    enable_metrics=True,   # so metrics get attached to the execution result
    num_shots=1250         # default number of shots (can be overridden below)
)

# |00> -> (H on qubit 0) -> (CX 0->1) -> Bell state (|00> + |11>)/sqrt(2)
qc_shots.add_gate(H, targets=0)
qc_shots.add_gate(CNOT, targets=[0, 1])

# Measure both qubits into classical bits c[0], c[1]
qc_shots.measure(qubit=0, cbit=0)
qc_shots.measure(qubit=1, cbit=1)

# Run the circuit multiple times ("shots")
shot_results = qc_shots.run_shots()


# Return counts and probabilities
counts = shot_results["counts"]
probs  = shot_results["probabilities"]

print("Counts (from run_shots return value):")
print(counts)
print("\nProbabilities:")
print(probs)


Counts (from run_shots return value):
{'00': 640, '11': 602, '01': 3, '10': 5}

Probabilities:
{'00': 0.512, '11': 0.4816, '01': 0.0024, '10': 0.004}


# Error Channels 

Implemented quantum noise channels:
- **Bit Flip Channel**: Flips |0⟩ ↔ |1⟩ with probability p
- **Phase Flip Channel**: Applies phase flip with probability p
- **Depolarizing Channel**: General noise model
- **Custom Kraus Operators**: User-defined error channels

In [33]:
# Initialize channel registry
chan_reg = ChannelRegistry()

print("Available channels:")
print(chan_reg.list())

# Demonstrate Bit Flip Channel
print("\n" + "="*60)
print("BIT FLIP CHANNEL")
print("="*60)
bit_flip_30 = chan_reg.get_param('bit_flip').instantiate(0.3)
print(f"{bit_flip_30}")
print(f"Number of Kraus operators: {len(bit_flip_30.kraus_ops)}")

# Apply bit flip to |0⟩ state multiple times to see stochastic behavior
print("\nApplying bit flip (p=0.3) to |0⟩ state:")
psi_0 = np.array([1.0, 0.0], dtype=complex)
for i in range(5):
    result = bit_flip_30.apply_statevector(psi_0.copy())
    print(f"Trial {i+1}: {result} -> measured as |{np.argmax(np.abs(result))}⟩")

# Demonstrate Phase Flip Channel  
print("\n" + "="*60)
print("PHASE FLIP CHANNEL (Bit-Phase Flip)")
print("="*60)
phase_flip_20 = chan_reg.get_param('bit_phase_flip').instantiate(0.2)
print(f"{phase_flip_20}")

# Apply to |+⟩ state = (|0⟩ + |1⟩)/√2
print("\nApplying phase flip (p=0.2) to |+⟩ state:")
psi_plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
print(f"Initial |+⟩ state: {psi_plus}")
for i in range(3):
    result = phase_flip_20.apply_statevector(psi_plus.copy())
    print(f"Trial {i+1}: {result}")

# Demonstrate Depolarizing Channel
print("\n" + "="*60)
print("DEPOLARIZING CHANNEL")
print("="*60)

depol_15 = chan_reg.get_param('depolarizing').instantiate(0.15)
print(f"{depol_15}")
# Apply to |+⟩ state = (|0⟩ + |1⟩)/√2
print("\nApplying depolarization (p=0.15) to |+⟩ state:")
psi_plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
print(f"Initial |+⟩ state: {psi_plus}")
for i in range(10):
    result = depol_15.apply_statevector(psi_plus.copy())
    print(f"Trial {i+1}: {result}")

Available channels:
['amplitude_damping', 'bit_flip', 'bit_phase_flip', 'depolarizing', 'phase_damping', 'phase_flip']

BIT FLIP CHANNEL
bit_flip (1q) Channel with 2 Kraus ops
Number of Kraus operators: 2

Applying bit flip (p=0.3) to |0⟩ state:
Trial 1: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 2: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 3: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 4: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 5: [0.+0.j 1.+0.j] -> measured as |1⟩

PHASE FLIP CHANNEL (Bit-Phase Flip)
bit_phase_flip (1q) Channel with 2 Kraus ops

Applying phase flip (p=0.2) to |+⟩ state:
Initial |+⟩ state: [0.70710678+0.j 0.70710678+0.j]
Trial 1: [0.-0.70710678j 0.+0.70710678j]
Trial 2: [0.-0.70710678j 0.+0.70710678j]
Trial 3: [0.70710678+0.j 0.70710678+0.j]

DEPOLARIZING CHANNEL
depolarizing (1q) Channel with 4 Kraus ops

Applying depolarization (p=0.15) to |+⟩ state:
Initial |+⟩ state: [0.70710678+0.j 0.70710678+0.j]
Trial 1: [0.70710678+0.j 0.70710678+0.j]
Trial 2: [0.70710678+0.j 0.707

# Noise Integration with Gates

In [36]:
# # Create a noisy Hadamard gate (10% bit flip error)
# bit_flip_10 = chan_reg.get_param('bit_flip').instantiate(0.1)
# noisy_h = Gate('noisy_h', gate_reg.get('h').matrix, noise=bit_flip_10)

# print("Comparing clean vs noisy Hadamard:")
# print("\nClean Hadamard on |0⟩:")
# psi_clean = np.array([1.0, 0.0], dtype=complex)
# result_clean = gate_reg.get('h').apply(psi_clean)
# print(f"Result: {result_clean}")
# print(f"Probabilities: {np.abs(result_clean)**2}")

# print("\nNoisy Hadamard on |0⟩ (with 10% bit flip):")
# # Run multiple times to see stochastic behavior
# results = []
# for i in range(5):
#     psi_noisy = np.array([1.0, 0.0], dtype=complex)
#     result = noisy_h.apply(psi_noisy)
#     results.append(result)
#     print(f"Trial {i+1}: {result}")

# QASM Parser

In [ ]:
from qasm_parser import parse_qasm_source

qasm = """
OPENQASM 2.0;
include "qelib1.inc";

qreg q[2];

h q[0];
x q[1];
cx q[0], q[1];
"""

instructions = parse_qasm_source(qasm)
print(instructions.items())

print("Parsed QASM instructions:")
for instr, value in instructions.items():
    if instr == "ops":
        print("ops:")
        for dict in value:
            for k, v in dict.items():
                print(f"\tGate {v['name']}: {}")
    else:
        print(f"{instr}: {value}")
    
        


dict_items([('n_qubits', 2), ('n_clbits', 0), ('ops', [{'name': 'h', 'qargs': [0], 'cargs': [], 'params': [], 'condition': None}, {'name': 'x', 'qargs': [1], 'cargs': [], 'params': [], 'condition': None}, {'name': 'cx', 'qargs': [0, 1], 'cargs': [], 'params': [], 'condition': None}])])
Parsed QASM instructions:
n_qubits: 2
n_clbits: 0
ops:
	name: h
	qargs: [0]
	cargs: []
	params: []
	condition: None
	name: x
	qargs: [1]
	cargs: []
	params: []
	condition: None
	name: cx
	qargs: [0, 1]
	cargs: []
	params: []
	condition: None
